# Market catalogue and pillar store

Conventions live in TOML. Pillars live in SQLite under a rates label: a name plus the
datetime they were saved, with one anchor date for every instrument on that snapshot.
`rates_environment` reads both and returns a `RatesEnvironment`. Loading a name without a
datetime returns the newest instance. The shipped catalogue is
`python/sjd/markets/catalogue.toml`: XAU, XPT, Dated Brent, and USDZAR, stored on the
`sample` label at midnight on 16 September 2026.

The environment prices from its curves. It does not hand the forward curve back, so each
market below is read through a zero-strike forward. That value is the discounted outright.


In [ ]:
import logging
import math
import sqlite3
import tempfile
from datetime import date
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import sjd
from sjd import markets
from sjd.markets import schema, specs, store

sjd.set_log_level("INFO")

anchor = markets.SAMPLE_ANCHOR
catalogue_text = Path(specs.CATALOGUE_PATH).read_text(encoding="utf-8")
xau_section = catalogue_text[
    catalogue_text.index("[markets.XAU]") : catalogue_text.index("[markets.XPT]")
]
print(specs.CATALOGUE_PATH)
print(xau_section.rstrip())

xau = markets.quote_spec("XAU", "USD")
xpt_zar = markets.quote_spec("XPT", "ZAR")
brent = markets.quote_spec("DATED_BRENT", "USD")
usdzar = markets.quote_spec("USDZAR", "ZAR")
assert isinstance(xau.forward, specs.GofoForward)
assert isinstance(xau.volatility, specs.DeltaVol)
assert isinstance(xau.dates, specs.PointDates)
assert xau.stores_spot
assert isinstance(xpt_zar.forward, specs.MetalTimesUsdZarForward)
assert not xpt_zar.stores_forward
assert isinstance(brent.forward, specs.OutrightForward)
assert isinstance(brent.volatility, specs.CalendarMonthMoneynessVol)
assert isinstance(brent.dates, specs.CalendarMonthDates)
assert isinstance(usdzar.forward, specs.OutrightForward)
assert isinstance(usdzar.dates, specs.PointDates)
print(markets.market_spec("XAU").summary)


## A market that exists only in markup

Adding a market is a catalogue entry plus its pillars. Nothing in the library changes.
`parse_catalogue` reads the text into the same spec objects as the shipped file. The database
is a temporary file, initialised from this catalogue rather than the shipped one.

The pillars belong to the rates label `silver`, saved at midnight on the anchor date. A
later save of the same name is a new datetime. Every instrument on one label shares that
label's anchor.


In [ ]:
silver_text = '''
[calendars.LONDON_METALS]
name = "UnitedKingdom.Metals"

[dates.LOCO_LONDON]
kind = "point"
calendar = "LONDON_METALS"
spot_lag_business_days = 2
expiry_before_value_business_days = 2

[discount.USD]
interpolation = "LINEAR"
extrapolate = false

[markets.XAG]
summary = "Loco London silver outright."

[markets.XAG.quotes.USD]
dates = "LOCO_LONDON"
forward = { kind = "outright", averaging = "POINT", interpolation = "LINEAR", extrapolate = false, roll = "FORWARD_VOLATILITY" }
volatility = { kind = "delta", interpolation = "BILINEAR", extrapolate = false }
'''
silver = markets.parse_catalogue(silver_text)
silver_quote = silver.quote_spec("XAG", "USD")
assert isinstance(silver_quote.forward, specs.OutrightForward)
assert silver_quote.forward.roll == specs.ForwardRoll.FORWARD_VOLATILITY
print(silver.market_spec("XAG").summary)

scratch = tempfile.TemporaryDirectory()
silver_path = Path(scratch.name) / "silver.db"
silver_connection = sqlite3.connect(silver_path)
schema.initialize(silver_connection, silver)
silver_as_of = f"{anchor.isoformat()}T00:00:00"
silver_connection.execute("PRAGMA foreign_keys = ON")
silver_connection.execute(
    "INSERT INTO rates_label (name, as_of, anchor_date) VALUES ('silver', ?, ?)",
    (silver_as_of, anchor.isoformat()),
)
silver_forward = (
    (date(2026, 9, 16), 30.0),
    (date(2026, 12, 16), 30.3),
    (date(2027, 9, 16), 31.2),
)
silver_connection.executemany(
    "INSERT INTO forward_pillar (label_name, as_of, market, currency, pillar_date, value) "
    "VALUES ('silver', ?, 'XAG', 'USD', ?, ?)",
    [(silver_as_of, day.isoformat(), value) for day, value in silver_forward],
)
silver_connection.executemany(
    "INSERT INTO discount_pillar (label_name, as_of, currency, pillar_date, discount_factor) "
    "VALUES ('silver', ?, 'USD', ?, ?)",
    [
        (silver_as_of, day.isoformat(), math.exp(-0.04 * (day - anchor).days / 365.0))
        for day, _value in silver_forward
    ],
)
silver_connection.executemany(
    "INSERT INTO volatility_pillar "
    "(label_name, as_of, market, currency, pillar_date, strike_dimension, volatility) "
    "VALUES ('silver', ?, 'XAG', 'USD', ?, ?, ?)",
    [
        (silver_as_of, day.isoformat(), delta, 0.25)
        for day, _value in silver_forward
        for delta in (10.0, 50.0, 90.0)
    ],
)
silver_connection.commit()
silver_market = markets.rates_environment(
    "XAG", anchor, "USD", silver_connection, catalogue=silver, label="silver"
)
silver_settlement = date(2026, 12, 16)
silver_contract = sjd.EuropeanForward(silver_settlement, 0.0, 1.0, sjd.BuySell.BUY)
sjd.require_valid(silver_contract, silver_market)
silver_discount = dict(store.discount_pillars(silver_connection, anchor, "USD"))[silver_settlement]
assert math.isclose(
    silver_contract.get_value(silver_market), 30.3 * silver_discount, rel_tol=1e-9
)
print(f"XAG outright value {silver_contract.get_value(silver_market):.6f}")


## The sample pillars

`load_sample` creates the tables from `schema.sql` and inserts one rates label, `sample`,
at midnight on the dummy anchor date. Every spot, forward, discount factor, and volatility
on that label uses that anchor. Another commodity is another quote on the same label.

The pillar tables have foreign keys to the quotes in the catalogue, so a row for a quote the
catalogue does not store is rejected. XAU ZAR and XPT ZAR forwards are not stored. Those
quotes are built from the USD metal curve and USDZAR.

Retrieval is `store.spot`, `forward_pillars`, `discount_pillars`, and `volatility_grid`.
Passing the anchor works here because only one label has that date. `store.rates_label`
selects the name, and omits the datetime for the newest instance.


In [ ]:
sample_path = Path(scratch.name) / "pillars.db"
connection = sqlite3.connect(sample_path)
assert schema.SCHEMA_VERSION == 3
loaded = markets.load_sample(connection)
assert loaded == anchor
sample_label = store.rates_label(connection, "sample")
assert sample_label.anchor == anchor
assert sample_label.as_of_text == f"{anchor.isoformat()}T00:00:00"
assert store.rates_label(connection, "sample", sample_label.as_of).name == "sample"
tables = {
    row[0]
    for row in connection.execute("SELECT name FROM sqlite_master WHERE type = 'table'")
}
assert {"rates_label", "spot", "forward_pillar", "discount_pillar", "volatility_pillar"} <= tables

xau_spot = store.spot(connection, anchor, "XAU", "USD")
xau_forwards = dict(store.forward_pillars(connection, anchor, "XAU", "USD"))
usd_discount = dict(store.discount_pillars(connection, anchor, "USD"))
zar_discount = dict(store.discount_pillars(connection, anchor, "ZAR"))
vol_dates, vol_strikes, vol_grid = store.volatility_grid(connection, anchor, "USDZAR", "ZAR")
assert xau_forwards[date(2026, 12, 16)] == 0.008
assert 50.0 in vol_strikes
assert len(vol_grid) == len(vol_strikes)
assert len(vol_grid[0]) == len(vol_dates)
print(f"label {sample_label.name} at {sample_label.as_of_text}, anchor {sample_label.anchor}")
print(f"XAU spot {xau_spot:.2f}")
print(f"December GOFO {xau_forwards[date(2026, 12, 16)]:.3f}")
print(f"USDZAR volatility dates {len(vol_dates)}, deltas {vol_strikes}")


## Four environments

XAU USD stores lease rates. The outright is the spot, grown by the lease rate and the two
discount factors from the spot date. XPT USD stores the outright itself. XPT ZAR is that
outright times USDZAR. Dated Brent is a calendar-month average, so a day in October reads the
October pillar.

The discount factors below are the ones just read from the database.


In [ ]:
class Collect(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []

    def emit(self, record):
        self.records.append(record)

market_logs = Collect()
core_logs = Collect()
logging.getLogger("sjd.markets.rates").addHandler(market_logs)
logging.getLogger("sjd._core").addHandler(core_logs)

def discounted_forward(market_name, currency, settlement):
    environment = markets.rates_environment(
        market_name, anchor, currency, connection
    )
    contract = sjd.EuropeanForward(settlement, 0.0, 1.0, sjd.BuySell.BUY)
    sjd.require_valid(contract, environment)
    return contract.get_value(environment)

horizon = date(2027, 9, 16)
october = date(2026, 10, 16)
usd_curve = sjd.DiscountCurve(anchor, list(usd_discount), list(usd_discount.values()))
spot_date = markets.spot_date(xau, anchor)
year_fraction = (date(2026, 12, 16) - spot_date).days / 360.0
gofo_outright = (
    xau_spot
    * usd_curve.get_discount_factor(spot_date)
    / usd_curve.get_discount_factor(date(2026, 12, 16))
    / (1.0 + xau_forwards[date(2026, 12, 16)] * year_fraction)
)
xau_value = discounted_forward("XAU", "USD", date(2026, 12, 16))
assert math.isclose(
    xau_value,
    gofo_outright * usd_discount[date(2026, 12, 16)],
    rel_tol=1e-9,
)

xpt_usd = dict(store.forward_pillars(connection, anchor, "XPT", "USD"))[horizon]
xpt_value = discounted_forward("XPT", "USD", horizon)
assert math.isclose(xpt_value, xpt_usd * usd_discount[horizon], rel_tol=1e-9)

usdzar_forward = dict(store.forward_pillars(connection, anchor, "USDZAR", "ZAR"))[horizon]
usdzar_value = discounted_forward("USDZAR", "ZAR", horizon)
assert math.isclose(usdzar_value, usdzar_forward * zar_discount[horizon], rel_tol=1e-9)

xpt_zar_value = discounted_forward("XPT", "ZAR", horizon)
assert math.isclose(xpt_zar_value, xpt_usd * usdzar_forward * zar_discount[horizon], rel_tol=1e-9)

brent_october = dict(store.forward_pillars(connection, anchor, "DATED_BRENT", "USD"))[
    date(2026, 10, 1)
]
brent_value = discounted_forward("DATED_BRENT", "USD", october)
assert math.isclose(brent_value, brent_october * usd_discount[october], rel_tol=1e-9)
print(f"XAU USD   {xau_value:,.2f}")
print(f"XPT USD   {xpt_value:,.2f}")
print(f"XPT ZAR   {xpt_zar_value:,.2f}")
print(f"USDZAR    {usdzar_value:,.2f}")
print(f"Brent USD {brent_value:,.2f}")


## Date conventions

A point quote's spot date is a number of business days after the anchor, and an option on a
value date expires a number of business days before it. Dated Brent has no spot lag. Its
pricing dates are the publication business days in the contract month, and cash settlement is
five business days after the last of them. A point quote has no contract month.


In [ ]:
assert spot_date == date(2026, 9, 18)
assert markets.option_expiry(xau, date(2026, 12, 29)) == date(2026, 12, 23)
december = markets.pricing_dates(brent, date(2026, 12, 10))
assert len(december) == 21
assert date(2026, 12, 25) not in december
assert december[-1] == date(2026, 12, 31)
assert markets.month_settlement(brent, date(2026, 12, 10)) == date(2027, 1, 8)
try:
    markets.spot_date(brent, anchor)
except markets.QuoteNotAvailable as error:
    print("Dated Brent:", error)
else:
    raise AssertionError("a calendar-month quote has no spot date")
try:
    markets.month_settlement(xau, anchor)
except markets.QuoteNotAvailable as error:
    print("Gold:", error)
else:
    raise AssertionError("a point quote has no contract month")


## What can go wrong

`PillarNotFound` is a label the database does not hold, or an anchor that belongs to no
label or to more than one. `RatesLabelError` is a save that would change a label's anchor
date. `QuoteNotAvailable` is a currency the catalogue does not quote. `CatalogueError` is
markup that does not parse. `SchemaVersionError` is a database written by an older schema.
A forward row for a quote the catalogue does not store fails the foreign key. SQLite forgets
`PRAGMA foreign_keys` on a new connection, so a writer turns it on itself. A calendar-month
market asked for a spot-volatility roll is a `ValueError`.


In [ ]:
try:
    store.spot(connection, date(2020, 1, 2), "XAU", "USD")
except store.PillarNotFound as error:
    assert isinstance(error, LookupError)
    print(error)
else:
    raise AssertionError("a missing anchor should be reported")

try:
    store.ensure_label(connection, "sample", f"{anchor.isoformat()}T00:00:00", date(2026, 9, 17))
except store.RatesLabelError as error:
    assert isinstance(error, ValueError)
    print(error)
else:
    raise AssertionError("a rates label keeps the anchor it was saved with")

try:
    markets.rates_environment("DATED_BRENT", anchor, "ZAR", connection)
except markets.QuoteNotAvailable as error:
    assert "ZAR" in str(error)
    print(error)
else:
    raise AssertionError("Dated Brent is quoted in USD only")

try:
    markets.parse_catalogue("this is not = a catalogue [[[")
except markets.CatalogueError as error:
    print("catalogue:", error)
else:
    raise AssertionError("broken markup should be a CatalogueError")

old = sqlite3.connect(":memory:")
old.executescript(
    "CREATE TABLE schema_info (id INTEGER PRIMARY KEY, version INTEGER NOT NULL);"
    "INSERT INTO schema_info VALUES (1, 1);"
)
try:
    schema.initialize(old)
except schema.SchemaVersionError as error:
    assert "schema version" in str(error)
    print(error)
else:
    raise AssertionError("an older schema should be refused")
old.close()

connection.execute("PRAGMA foreign_keys = ON")
try:
    connection.execute(
        "INSERT INTO forward_pillar (label_name, as_of, market, currency, pillar_date, value) "
        "VALUES ('sample', ?, 'XAG', 'USD', ?, ?)",
        (f"{anchor.isoformat()}T00:00:00", horizon.isoformat(), 25.0),
    )
except sqlite3.IntegrityError as error:
    assert "FOREIGN KEY" in str(error).upper()
    print(error)
else:
    raise AssertionError("XAG is not a stored forward in the shipped catalogue")

try:
    markets.rates_environment(
        "DATED_BRENT",
        anchor,
        "USD",
        connection,
        roll=specs.ForwardRoll.SPOT_VOLATILITY,
    )
except ValueError as error:
    assert "spot-volatility" in str(error)
    print(error)
else:
    raise AssertionError("Dated Brent has no spot-volatility roll")


## What it logs

Each `rates_environment` call writes `loaded market` at `INFO` on `sjd.markets.rates`, with
the market, the currency, the rates label, the datetime, the anchor, and the roll.

Loading validates the curves, so warnings from the pillars show up as well. XAU's lease rates
have no pillar on the anchor, and that logs a flat back-fill. Dated Brent's surface allows
extrapolation, and that logs too.


In [ ]:
messages = [record.getMessage() for record in market_logs.records]
assert all(record.levelno == logging.INFO for record in market_logs.records)
assert any(message.startswith("loaded market XAU/USD") for message in messages)
assert any(message.startswith("loaded market XPT/ZAR") for message in messages)
assert any(message.startswith("loaded market DATED_BRENT/USD") for message in messages)
assert any(message.startswith("loaded market USDZAR/ZAR") for message in messages)
assert all("label sample at" in message for message in messages)
assert all("roll DETERMINISTIC" in message for message in messages)
core_messages = [record.getMessage() for record in core_logs.records]
assert any("flat through the first observation" in message for message in core_messages)
assert any("Extrapolation past the last pillar" in message for message in core_messages)
for message in messages:
    print(message)
for record in core_logs.records:
    print(record.levelname, record.getMessage())


## Save and reload

The conventions are the TOML file, read at import. The numbers are the SQLite file written
above. Closing it and opening it again returns the same spot. The temporary directory is the
only place this notebook writes.


In [ ]:
connection.close()
silver_connection.close()
reopened = sqlite3.connect(sample_path)
assert store.spot(reopened, anchor, "XAU", "USD") == xau_spot
assert dict(store.forward_pillars(reopened, anchor, "USDZAR", "ZAR"))[horizon] == usdzar_forward
reopened.close()
print("reloaded", sample_path.name)
